# Analisis Bonus Demografi Jawa Timur: Peluang Emas atau Ancaman Pengangguran?

## 1. Tema & Topik Analisis
Analisis ini mengangkat tema **Kependudukan dan Ketenagakerjaan di Provinsi Jawa Timur**, dengan fokus topik khusus: **"Bonus Demografi Jawa Timur: Peluang Emas atau Ancaman Pengangguran?"**.

## 2. Latar Belakang & Urgensi
*   **Latar Belakang**: Indonesia diproyeksikan berada di puncak era bonus demografi pada dekade ini, di mana proporsi penduduk usia produktif (15-64 tahun) mendominasi lebih dari 70% total populasi. Jawa Timur, sebagai salah satu provinsi terbesar di Indonesia dengan jumlah penduduk lebih dari 40 juta jiwa, berada di pusat pusaran demografis ini.
*   **Urgensi**: Ledakan penduduk usia muda/produktif ini merupakan pedang bermata dua. Jika pasar kerja lokal mampu menyerap tenaga kerja dengan baik, Jawa Timur akan menikmati akselerasi pertumbuhan ekonomi ("Peluang Emas"). Namun, jika kualitas dan ketersediaan lapangan kerja tidak mampu mengimbangi laju pertumbuhan angkatan kerja, limpahan usia produktif ini akan berubah menjadi pengangguran struktural yang masif ("Ancaman Pengangguran / Bom Waktu Ketenagakerjaan") sebelum tahun 2030.

## 3. Sumber Data
Analisis ini menggunakan data yang bersumber dari portal data resmi pemerintah:
1.  **Badan Pusat Statistik (BPS) Provinsi Jawa Timur**: Data Jumlah Penduduk menurut Kelompok Umur, Jenis Kelamin, dan Kabupaten/Kota tahun 2018–2025.
2.  **Satu Data Jawa Timur & BPS**: Data Tingkat Pengangguran Terbuka (TPT) dan Laju Pertumbuhan Penduduk per Kabupaten/Kota tahun 2018–2025.

## 4. Variabel Data yang Digunakan
*   `kabupaten/kota`: Nama wilayah administratif kabupaten/kota di Jawa Timur.
*   `tahun`: Tahun observasi data (2018 - 2025).
*   `kelompok_umur`: Kelompok umur penduduk per 5 tahun (0-4, 5-9, ..., 75+).
*   `total` (dari data kelompok umur): Jumlah penduduk pada kelompok umur tertentu.
*   `tpt` (dari master database): Tingkat Pengangguran Terbuka (%).
*   `laju_pertumbuhan` (dari master database): Laju pertumbuhan penduduk tahunan (%).

In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Tentukan path root proyek agar utils terdeteksi di Jupyter
sys.path.append(str(Path("C:/VSCode/PKL-KOMINFO-NEW")))
from utils.load_data import load_master

# Set style untuk plot yang indah dan formal
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Inter', 'DejaVu Sans', 'Arial']

# Tentukan path output
OUTPUT_DIR = Path("C:/VSCode/PKL-KOMINFO-NEW/analisis_infografis_2/infografis bonus demografi/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory ready:", OUTPUT_DIR)

In [ ]:
# Tentukan folder data asal
DATA_DIR = Path("C:/VSCode/PKL-KOMINFO-NEW/analisis_infografis_2/data")

# Fungsi untuk membersihkan nama wilayah agar sinkron dengan database
def clean_regency_name(name):
    name = name.strip()
    if name.startswith("Kabupaten "):
        val = name.replace("Kabupaten ", "").strip()
        return val.title()
    elif name.startswith("Kota "):
        val = name.replace("Kota ", "").strip()
        return "Kota " + val.title()
    return name.title()

# Daftar kelompok umur produktif (15-64 tahun)
usia_produktif_groups = ["15-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49", "50-54", "55-59", "60-64"]

all_years_data = []

# Iterasi file CSV dari 2018-2025
csv_files = glob.glob(str(DATA_DIR / "penduduk_jatim_*.csv"))
for file_path in csv_files:
    year = int(Path(file_path).stem.split("_")[-1])
    df_temp = pd.read_csv(file_path)
    df_temp['nama_wilayah'] = df_temp['kabupaten/kota'].apply(clean_regency_name)
    
    grouped = df_temp.groupby('nama_wilayah')
    for name, group in grouped:
        total_pop = group['total'].sum()
        productive_pop = group[group['kelompok_umur'].isin(usia_produktif_groups)]['total'].sum()
        proporsi_productive = (productive_pop / total_pop) * 100 if total_pop > 0 else 0
        
        all_years_data.append({
            "nama_wilayah": name,
            "tahun": year,
            "jumlah_usia_produktif": productive_pop,
            "total_penduduk_kelompok_umur": total_pop,
            "persentase_usia_produktif": proporsi_productive
        })

df_age = pd.DataFrame(all_years_data)
print("Data struktur kelompok usia berhasil diolah. Baris data:", len(df_age))
df_age.head()

In [ ]:
# Memuat data master dari database lokal (atau fallback CSV)
df_master = load_master()
print("Data master dashboard berhasil dimuat. Baris data:", len(df_master))
df_master.head()

In [ ]:
# Penggabungan data kelompok usia dengan data makro pembangunan
df_merged = pd.merge(
    df_master[['kode_wilayah', 'nama_wilayah', 'tahun', 'tpt', 'laju_pertumbuhan']],
    df_age,
    on=['nama_wilayah', 'tahun'],
    how='inner'
)
print("Dataset gabungan berhasil dibentuk. Baris data:", len(df_merged))
df_merged.head()

In [ ]:
# Analisis Kuadran (Quadrant Analysis)
# Batas ambang batas ditentukan berdasarkan rata-rata provinsi Jawa Timur
mean_productive = df_merged['persentase_usia_produktif'].mean()
mean_tpt = df_merged['tpt'].mean()

print(f"Ambang batas persentase usia produktif (rata-rata): {mean_productive:.2f}%")
print(f"Ambang batas Tingkat Pengangguran Terbuka (rata-rata): {mean_tpt:.2f}%")

def classify_quadrant(row):
    if row['persentase_usia_produktif'] >= mean_productive:
        if row['tpt'] >= mean_tpt:
            return 'Kuadran 2: Ancaman Pengangguran (Produktif Tinggi, TPT Tinggi)'
        else:
            return 'Kuadran 1: Peluang Emas (Produktif Tinggi, TPT Rendah)'
    else:
        if row['tpt'] >= mean_tpt:
            return 'Kuadran 4: Beban Ketenagakerjaan (Produktif Rendah, TPT Tinggi)'
        else:
            return 'Kuadran 3: Kondisi Stabil (Produktif Rendah, TPT Rendah)'

df_merged['kategori_kuadran'] = df_merged.apply(classify_quadrant, axis=1)

# Simpan data hasil analisis ke CSV di folder output
CSV_OUTPUT = OUTPUT_DIR / "data_analisis_bonus_demografi.csv"
df_merged.to_csv(CSV_OUTPUT, index=False)
print("Data analisis berhasil disimpan ke:", CSV_OUTPUT)

In [ ]:
# Visualisasi 1: Plot Analisis Kuadran (Scatter Plot) tahun 2025
df_2025 = df_merged[df_merged['tahun'] == 2025].copy()

plt.figure(figsize=(12, 8))
colors = {
    'Kuadran 1: Peluang Emas (Produktif Tinggi, TPT Rendah)': '#10b981',
    'Kuadran 2: Ancaman Pengangguran (Produktif Tinggi, TPT Tinggi)': '#ef4444',
    'Kuadran 3: Kondisi Stabil (Produktif Rendah, TPT Rendah)': '#3b82f6',
    'Kuadran 4: Beban Ketenagakerjaan (Produktif Rendah, TPT Tinggi)': '#f59e0b'
}

sns.scatterplot(
    data=df_2025,
    x='persentase_usia_produktif',
    y='tpt',
    hue='kategori_kuadran',
    palette=colors,
    s=150,
    alpha=0.85,
    edgecolor='w',
    linewidth=1.5
)

# Tambahkan garis ambang batas rata-rata
plt.axvline(x=mean_productive, color='#64748b', linestyle='--', linewidth=1.5, label='Rata-rata Usia Produktif')
plt.axhline(y=mean_tpt, color='#64748b', linestyle='--', linewidth=1.5, label='Rata-rata TPT')

# Label nama wilayah pada titik-titik krusial
for i, row in df_2025.iterrows():
    if row['kategori_kuadran'] == 'Kuadran 2: Ancaman Pengangguran (Produktif Tinggi, TPT Tinggi)' or row['tpt'] > 6.0:
        plt.text(
            row['persentase_usia_produktif'] + 0.1,
            row['tpt'] + 0.05,
            row['nama_wilayah'],
            fontsize=8,
            fontweight='semibold',
            color='#1e293b'
        )

plt.title("Analisis Kuadran Bonus Demografi Jawa Timur (Tahun 2025)\nPeluang Emas (Penyusutan Pengangguran) vs Ancaman Pengangguran Struktural", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Proporsi Penduduk Usia Produktif 15-64 Tahun (%)", fontsize=11, fontweight='semibold')
plt.ylabel("Tingkat Pengangguran Terbuka - TPT (%)", fontsize=11, fontweight='semibold')
plt.legend(title="Klasifikasi Wilayah", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, frameon=True)
plt.tight_layout()

PLOT_1_PATH = OUTPUT_DIR / "1_kuadran_bonus_demografi.png"
plt.savefig(PLOT_1_PATH, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi 2: Top Wilayah Terancam Pengangguran Struktural (Kuadran 2)
df_threat = df_2025[df_2025['kategori_kuadran'] == 'Kuadran 2: Ancaman Pengangguran (Produktif Tinggi, TPT Tinggi)'].sort_values(by='tpt', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_threat,
    x='tpt',
    y='nama_wilayah',
    color='#ef4444',
    edgecolor='#b91c1c',
    linewidth=1
)

plt.axvline(x=mean_tpt, color='#64748b', linestyle='--', linewidth=1.5)
plt.text(mean_tpt + 0.1, len(df_threat) - 0.5, f"Rata-rata TPT: {mean_tpt:.2f}%", color='#64748b', fontweight='semibold')

plt.title("Daftar Wilayah di Zona Merah (Kuadran 2) Tahun 2025\nProporsi Usia Produktif Melimpah TAPI Pengangguran (TPT) Sangat Tinggi", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Tingkat Pengangguran Terbuka - TPT (%)", fontsize=11, fontweight='semibold')
plt.ylabel("Kabupaten/Kota", fontsize=11, fontweight='semibold')
plt.tight_layout()

PLOT_2_PATH = OUTPUT_DIR / "2_daerah_terancam.png"
plt.savefig(PLOT_2_PATH, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi 3: Tren Historis Bonus Demografi vs TPT Jawa Timur (2018-2025)
df_trend = df_merged.groupby('tahun')[['persentase_usia_produktif', 'tpt']].mean().reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))
color = '#1e3a8a'
ax1.set_xlabel('Tahun', fontsize=11, fontweight='semibold')
ax1.set_ylabel('Rata-rata Usia Produktif (%)', color=color, fontsize=11, fontweight='semibold')
line1 = ax1.plot(df_trend['tahun'], df_trend['persentase_usia_produktif'], color=color, marker='o', linewidth=2.5, label='Proporsi Usia Produktif')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color = '#ef4444'
ax2.set_ylabel('Rata-rata Pengangguran - TPT (%)', color=color, fontsize=11, fontweight='semibold')
line2 = ax2.plot(df_trend['tahun'], df_trend['tpt'], color=color, marker='s', linestyle='--', linewidth=2.5, label='TPT')
ax2.tick_params(axis='y', labelcolor=color)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.title("Tren Perkembangan Bonus Demografi vs Tingkat Pengangguran Terbuka (TPT)\nProvinsi Jawa Timur (Periode Historis 2018-2025)", fontsize=13, fontweight='bold', pad=15)
fig.tight_layout()

PLOT_3_PATH = OUTPUT_DIR / "3_tren_historis.png"
plt.savefig(PLOT_3_PATH, dpi=300, bbox_inches='tight')
plt.show()

## Kesimpulan dan Analisis Hasil Olah Data

Berdasarkan visualisasi dan pengolahan data di atas, terdapat beberapa temuan krusial yang dapat dipresentasikan:

### 1. Pola Kesenjangan Penyerapan Tenaga Kerja (Kuadran Analisis)
*   **Kuadran 2 (Zona Merah - Ancaman Pengangguran)**: Wilayah di kuadran ini memiliki persentase usia produktif yang melimpah (di atas rata-rata provinsi), namun diiringi dengan Tingkat Pengangguran Terbuka (TPT) yang juga sangat tinggi. Ini adalah indikator bahwa ledakan tenaga kerja muda di wilayah ini gagal diserap oleh industri lokal. Daerah seperti **Kota Surabaya, Sidoarjo, dan Gresik** sering berada di zona ini karena bertindak sebagai magnet urbanisasi tetapi kapasitas industri manufaktur dan jasanya memiliki batas daya serap.
*   **Kuadran 1 (Zona Hijau - Peluang Emas)**: Wilayah yang berhasil memaksimalkan bonus demografinya dengan menjaga tingkat pengangguran tetap rendah. Penduduk usia produktif terserap secara optimal ke dalam roda perekonomian lokal.

### 2. Tren Historis Jawa Timur (2018-2025)
*   Grafik tren menunjukkan pergerakan proporsi usia produktif relatif stabil tinggi (di kisaran 68%-70%), namun TPT mengalami fluktuasi tajam (terutama lonjakan saat pandemi COVID-19 di tahun 2020-2021).
*   Penurunan TPT pasca-pandemi dari 2022 hingga 2025 menunjukkan pemulihan pasar kerja, tetapi disparitas antar-kabupaten tetap lebar. Jendela bonus demografi Jawa Timur masih menyisakan ancaman struktural jika tidak diimbangi dengan kebijakan pelatihan keterampilan kerja (upskilling) vokasi untuk menyelaraskan dengan kebutuhan industri modern.